# Silver: PDFs to Markdown

Turns the PDFs in `bronze_pdfs` into one Markdown document per PDF. It runs as the `silver_parser` task of the job `ingest_financebench`.

```
bronze_pdfs --stream: ai_parse_document + check--> silver_parsed      (good parses)
                                          \--> silver_log             (bad parses: reason, retryable?)
silver_log --retry the temporary failures--> silver_parsed
silver_parsed --ai_query, one page at a time--> silver_pages          (Markdown of each page)
silver_pages --view--> silver_documents                              (one Markdown document per PDF)
```

| Table / view | Holds |
| --- | --- |
| `silver_parsed` | One row per PDF whose parse is complete: the parsed elements (VARIANT) and page counts |
| `silver_pages` | The Markdown of each converted page |
| `silver_log` | The trail: every failed parse and every failed page, with the reason, the attempt number and whether a retry makes sense |
| `silver_documents` (view) | The final Markdown per PDF, only for PDFs whose pages are all converted |
| `silver_failures` (view) | What is still open: PDFs not parsed and pages not converted |

**The rules that keep it simple**

1. **Check every result, and keep the bad ones apart.** A parse that is not complete, or a page that did not convert, never goes into the good tables. It goes to `silver_log`.
2. **Every stage can run again.** Each stage works out what is still missing and does only that, so a rerun, a retry and a crash recovery are the same code.
3. **The code, not the model, owns the document structure.** The model only converts the text of one page. The page markers and the `---` between pages are added by a view.
4. **Small steps with a budget.** Every stage has a limit per run, so a run on a limited quota (the Databricks free tier) cannot spend more than you allow.

| Widget | Default | Meaning |
| --- | --- | --- |
| `catalog`, `schema` | `workspace`, `finance_bench_rag` | From the bundle variables |
| `endpoint` | `databricks-qwen3-next-80b-a3b-instruct` | Model endpoint used by `ai_query` |
| `max_files_per_batch` | `2` | Parse stream: bronze files per micro-batch. Small, so the parser's rate limit is not hit |
| `max_attempts` | `3` | A PDF or page is given up on after this many failed attempts |
| `retry_page_budget` | `1000` | PDF pages that retrying failed parses may cost in one run. `0` = no retries |
| `retry_pause_seconds` | `10` | Pause between retried PDFs |
| `max_pages_per_run` | `1000` | Pages converted with `ai_query` in one run |
| `pages_per_chunk` | `200` | Pages per save point while converting |

In [ ]:
import json
import re
import time

from delta.tables import DeltaTable
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("endpoint", "databricks-qwen3-next-80b-a3b-instruct")
dbutils.widgets.text("max_files_per_batch", "2")
dbutils.widgets.text("max_attempts", "3")
dbutils.widgets.text("retry_page_budget", "1000")
dbutils.widgets.text("retry_pause_seconds", "10")
dbutils.widgets.text("max_pages_per_run", "1000")
dbutils.widgets.text("pages_per_chunk", "200")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
endpoint = dbutils.widgets.get("endpoint")
max_files_per_batch = int(dbutils.widgets.get("max_files_per_batch"))
max_attempts = int(dbutils.widgets.get("max_attempts"))
retry_page_budget = int(dbutils.widgets.get("retry_page_budget"))
retry_pause_seconds = int(dbutils.widgets.get("retry_pause_seconds"))
max_pages_per_run = int(dbutils.widgets.get("max_pages_per_run"))
pages_per_chunk = int(dbutils.widgets.get("pages_per_chunk"))

bronze = f"{catalog}.{schema}.bronze_pdfs"
parsed_tbl = f"{catalog}.{schema}.silver_parsed"
pages_tbl = f"{catalog}.{schema}.silver_pages"
log_tbl = f"{catalog}.{schema}.silver_log"
docs_view = f"{catalog}.{schema}.silver_documents"
failures_view = f"{catalog}.{schema}.silver_failures"
checkpoint = f"/Volumes/{catalog}/{schema}/docs/_checkpoints/silver_parse"

## 1. Create the tables and views

Explicit `CREATE ... IF NOT EXISTS`, in a cell of its own. Safe to rerun.

- `silver_log` is append-only. A PDF that fails twice has two rows (attempt 1 and 2), so the log is a history, and the attempt number is how a retry knows when to stop.
- `silver_documents` puts the document together: each page gets its `<!-- page N -->` marker (N is the real PDF page number), the pages are ordered, and they are joined with `---`. A document appears only when **all** its pages are converted.
- `silver_failures` lists what is still open. A failed parse is open until the PDF is in `silver_parsed`. A failed page is open until it is in `silver_pages`.

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {parsed_tbl} (
  path STRING, parsed VARIANT, pdf_pages INT, pages INT, parsed_at TIMESTAMP
)""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {pages_tbl} (
  path STRING, page_no INT, markdown STRING, converted_at TIMESTAMP
)""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {log_tbl} (
  path STRING, stage STRING, page_no INT, attempt INT, retryable BOOLEAN, reason STRING, pdf_pages INT, logged_at TIMESTAMP
)""")

spark.sql(f"""
CREATE OR REPLACE VIEW {docs_view} AS
SELECT p.path, g.markdown, p.pages
FROM {parsed_tbl} p
JOIN (
  SELECT path, count(*) AS n,
         concat_ws('\\n\\n---\\n\\n', transform(array_sort(collect_list(struct(page_no, concat('<!-- page ', page_no, ' -->\\n', markdown)))), x -> x.col2)) AS markdown
  FROM {pages_tbl} GROUP BY path
) g ON g.path = p.path AND g.n = p.pages
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {failures_view} AS
SELECT l.path, l.stage, l.page_no, max(l.attempt) AS attempts,
       max_by(l.retryable, l.attempt) AS retryable, max_by(l.reason, l.attempt) AS reason, max(l.pdf_pages) AS pdf_pages
FROM {log_tbl} l
LEFT JOIN {parsed_tbl} p ON l.stage = 'parse' AND p.path = l.path
LEFT JOIN {pages_tbl} g ON l.stage = 'convert' AND g.path = l.path AND g.page_no = l.page_no
WHERE p.path IS NULL AND g.path IS NULL
GROUP BY l.path, l.stage, l.page_no
""")


def insert_new(df, table, keys):
    """Insert the rows whose key is not in the table yet. Running it twice with the same rows changes nothing."""
    condition = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
    DeltaTable.forName(spark, table).alias("t").merge(df.alias("s"), condition).whenNotMatchedInsertAll().execute()

## 2. Parse the PDFs (stream) and check each parse

A stream reads `bronze_pdfs`. For each micro-batch, `parse_docs` runs `ai_parse_document` (version 2.0) **once** per PDF and checks the result in plain Python (`check_parse`):

| Result of the parse | What happens |
| --- | --- |
| Complete: has elements and no page errors | Stored in `silver_parsed` |
| Some pages failed (`Request rate limit exceeded`, `Internal error`) | Logged as **retryable**. The PDF is not stored, because a document with missing pages would silently lose text |
| No elements: the PDF is over 500 pages, or the conversion failed | Logged as **not retryable**. Retrying cannot help |

**Using the stream's error handling.** A stream remembers in its checkpoint which micro-batches finished. If a batch fails, the checkpoint does not move, so the next run starts again at that batch. Nothing is lost and nothing is skipped. The one thing a retried batch must not do is parse the same PDFs again (that is billed), so each batch first drops the PDFs that are already in `silver_parsed` or `silver_log`. Together with `maxFilesPerTrigger` (small batches, few PDFs at once), this gives a restart that costs almost nothing.

**Challenge: `ai_parse_document` answers some pages with an error.** In the first full run 3,180 of 49,965 pages (6%), in 55 PDFs, failed with `Request rate limit exceeded` or `Internal error`, because 10 PDFs were parsed at once. **Solution:** check every parse, never store an incomplete one, parse 2 files at a time, and retry the failures later (step 3).

In [ ]:
PERMANENT = re.compile(r"exceeds the limit|conversion failed", re.I)


def check_parse(doc):
    """None if the parse is complete, else (reason, retryable)."""
    errors = doc.get("error_status") or []
    if not doc["document"]["elements"]:
        reason = errors[0]["error_message"] if errors else "no elements"
        return reason, not PERMANENT.search(reason)
    if errors:
        return f"{len(errors)} pages failed: {errors[0]['error_message']}", True
    return None


def parse_docs(df):
    """Parse the PDFs in `df` (path, content). Complete parses go to silver_parsed, the others to silver_log."""
    rows = df.select(
        "path", F.expr("CAST(ai_parse_document(content, map('version', '2.0')) AS STRING)").alias("json")
    ).collect()
    attempts = {
        r.path: r.n
        for r in spark.table(log_tbl).filter("stage = 'parse'").groupBy("path").agg(F.max("attempt").alias("n")).collect()
    }

    good, bad = [], []
    for r in rows:
        doc = json.loads(r.json)
        pdf_pages = len(doc["document"]["pages"])
        problem = check_parse(doc)
        if problem is None:
            pages = len({(e.get("bbox") or [{"page_id": 0}])[0]["page_id"] for e in doc["document"]["elements"]})
            good.append((r.path, r.json, pdf_pages, pages))
        else:
            reason, retryable = problem
            bad.append((r.path, "parse", None, attempts.get(r.path, 0) + 1, retryable, reason[:500], pdf_pages))

    if good:
        parsed = spark.createDataFrame(good, "path string, j string, pdf_pages int, pages int").select(
            "path", F.expr("parse_json(j)").alias("parsed"), "pdf_pages", "pages", F.current_timestamp().alias("parsed_at")
        )
        insert_new(parsed, parsed_tbl, ["path"])
    if bad:
        logged = spark.createDataFrame(
            bad, "path string, stage string, page_no int, attempt int, retryable boolean, reason string, pdf_pages int"
        ).withColumn("logged_at", F.current_timestamp())
        insert_new(logged, log_tbl, ["path", "stage", "page_no", "attempt"])
    return len(good), len(bad)


def parse_new_pdfs(batch_df, batch_id):
    """One micro-batch: skip the PDFs that are already recorded (a retried batch costs nothing), parse the rest."""
    known = spark.table(parsed_tbl).select("path").union(spark.table(log_tbl).filter("stage = 'parse'").select("path"))
    good, bad = parse_docs(batch_df.join(known, "path", "left_anti"))
    print(f"batch {batch_id}: {good} parsed, {bad} logged as bad")


(
    spark.readStream.option("maxFilesPerTrigger", max_files_per_batch).table(bronze)
    .writeStream.foreachBatch(parse_new_pdfs)
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .start().awaitTermination()
)

## 3. Retry the failed parses

Takes the failed parses that are **retryable** and have attempts left (from `silver_failures`) and parses them again with the **same function**, `parse_docs`. There is no special repair code: a PDF that now parses completely goes to `silver_parsed`, and one that fails again gets another row in the log with the next attempt number.

Built for a limited quota:

- **Budget.** `retry_page_budget` (default 1000) is the number of PDF pages one run may spend on retries. With `0` nothing is retried. The cheapest (shortest) PDFs go first, so the budget fixes the most documents.
- **One PDF at a time, with a pause** (`retry_pause_seconds`), so the rate limit is not hit again.
- **Stop when the service is limiting.** Two failures in a row stop the retries for this run.
- **Attempt cap.** After `max_attempts` a PDF is left alone. It stays in `silver_failures`, where you can see it.
- **Never retried:** failures that cannot succeed (a PDF over 500 pages, a failed conversion).

In [ ]:
candidates = spark.sql(f"""
SELECT path, pdf_pages FROM {failures_view}
WHERE stage = 'parse' AND retryable AND attempts < {max_attempts}
ORDER BY pdf_pages
""").collect()
print(f"{len(candidates)} PDFs have a retryable parse failure; page budget {retry_page_budget}.")

spent, in_a_row = 0, 0
for c in candidates:
    if spent + c.pdf_pages > retry_page_budget or in_a_row >= 2:  # never go over the budget
        break
    good, bad = parse_docs(spark.table(bronze).filter(f"path = '{c.path}'"))
    spent += c.pdf_pages
    in_a_row = in_a_row + 1 if bad else 0
    print(f"{c.path.split('/')[-1]}: {'parsed' if good else 'failed again'} ({c.pdf_pages} pages)")
    time.sleep(retry_pause_seconds)
print(f"Retry pages spent: {spent} of {retry_page_budget}." + (" Stopped: the service keeps failing." if in_a_row >= 2 else ""))

## 4. Convert the pages to Markdown with `ai_query`

One `ai_query` call per **page**. For every PDF in `silver_parsed`, the pages that are not in `silver_pages` yet (and have attempts left) are converted. Each result is checked in Python: an error or an empty answer is logged as a failed page, and anything else is stored.

**Why one page per call:**

- **The output limit.** The model endpoint returns at most 10,000 tokens per call. A whole filing in one call was silently cut off (a 150-page filing came back at about 43,000 characters). A page always fits.
- **Nothing to repair.** The model only converts the text of one page. It is told not to write page markers or separators, and the code removes any it writes anyway (it sometimes mistook the printed page number, such as "17", for a marker). The marker and the `---` are added by the `silver_documents` view. In an earlier version the model wrote them, 151 of 359 documents had repeated or missing markers, and the repair code was most of the notebook.
- **Small retries.** A failed page costs one call to retry, not a whole document.

The page number is the real PDF page number (the parser's page id plus 1).

**Budget and save points.** `max_pages_per_run` (default 1000) limits the calls of one run. Pages are converted in chunks of `pages_per_chunk` (default 200) and each chunk is saved before the next, so a crash loses at most one chunk. A chunk that stores nothing stops the run.

Total tokens are about the same as converting several pages per call. Only the short prompt repeats for every page.

In [ ]:
PROMPT = """Convert the parsed elements of ONE page of a financial filing (PDF) into clean Markdown.

The input is a JSON array in reading order. Each element has a "type" (title, section_header, text, table, list_item, caption, footnote, figure, page_header, page_footer, page_number, ...) and its "content" (tables are HTML).

Rules:
- Keep ALL content exactly as written. Do not summarize, reword, reorder or add anything. Copy numbers, names and punctuation exactly.
- A "title" becomes "# Title". A "section_header" becomes "## Heading" (use "###" for clearly nested sections). Never turn normal text into a heading.
- Keep plain text as paragraphs, and lists as Markdown lists.
- Convert tables to Markdown pipe tables with a header row and a "| --- |" row. Keep every row and column.
- Keep page headers, page footers and page numbers as italic lines (*like this*).
- Output ONLY the Markdown of this page. Do not write page markers, "---" separators, code fences or commentary.

Elements:
"""
MARKER = re.compile(r"<!--.*?-->", re.S)


def pages_to_convert(limit):
    """Pages of parsed PDFs that are not converted yet and have attempts left."""
    pages = spark.sql(f"""
    SELECT d.path, coalesce(e.value:bbox[0].page_id::INT, 0) + 1 AS page_no,
           e.pos, e.value:type::STRING AS type, e.value:content::STRING AS content
    FROM {parsed_tbl} d
    LEFT JOIN (SELECT path, count(*) AS n FROM {pages_tbl} GROUP BY path) g ON g.path = d.path
    , LATERAL variant_explode(d.parsed:document:elements) e
    WHERE coalesce(g.n, 0) < d.pages
    """).groupBy("path", "page_no").agg(
        F.to_json(F.transform(F.array_sort(F.collect_list(F.struct("pos", "type", "content"))),
                              lambda x: F.struct(x["type"], x["content"]))).alias("elements")
    )
    done = spark.table(pages_tbl).select("path", "page_no")
    given_up = (
        spark.table(log_tbl).filter("stage = 'convert'").groupBy("path", "page_no").count()
        .filter(f"count >= {max_attempts}").select("path", "page_no")
    )
    return pages.join(done, ["path", "page_no"], "left_anti").join(given_up, ["path", "page_no"], "left_anti").limit(limit)


def convert_chunk(limit):
    """Convert up to `limit` pages. Good pages go to silver_pages, bad ones to silver_log. Returns (stored, failed)."""
    rows = (
        pages_to_convert(limit)
        .withColumn("prompt", F.concat(F.lit(PROMPT), F.col("elements")))
        .repartition(32)  # run the calls in parallel
        .select("path", "page_no", F.expr(
            f"ai_query('{endpoint}', prompt, modelParameters => named_struct('temperature', 0.0, 'max_tokens', 10000), failOnError => false)"
        ).alias("r"))
        .collect()
    )
    attempts = {
        (r.path, r.page_no): r.n
        for r in spark.table(log_tbl).filter("stage = 'convert'").groupBy("path", "page_no").agg(F.max("attempt").alias("n")).collect()
    }
    good, bad = [], []
    for r in rows:
        text = MARKER.sub("", r.r.result or "").strip()
        if r.r.errorMessage or not text:
            reason = (r.r.errorMessage or "empty answer")[:500]
            bad.append((r.path, "convert", r.page_no, attempts.get((r.path, r.page_no), 0) + 1, True, reason, None))
        else:
            good.append((r.path, r.page_no, text))
    if good:
        insert_new(spark.createDataFrame(good, "path string, page_no int, markdown string").withColumn("converted_at", F.current_timestamp()),
                   pages_tbl, ["path", "page_no"])
    if bad:
        insert_new(spark.createDataFrame(bad, "path string, stage string, page_no int, attempt int, retryable boolean, reason string, pdf_pages int")
                   .withColumn("logged_at", F.current_timestamp()), log_tbl, ["path", "stage", "page_no", "attempt"])
    return len(good), len(bad)


spent = 0
while spent < max_pages_per_run:
    stored, failed = convert_chunk(min(pages_per_chunk, max_pages_per_run - spent))
    spent += stored + failed
    print(f"{stored} pages stored, {failed} failed")
    if stored == 0:
        break
print(f"Pages converted in this run: {spent} of a budget of {max_pages_per_run}.")

## 5. Check the result

What is stored and what is still open. `silver_failures` is the list to look at: each row is a PDF that is not parsed, or a page that is not converted, with the reason, the number of attempts and whether a retry makes sense.

In [ ]:
counts = {
    "PDFs parsed (silver_parsed)": spark.table(parsed_tbl).count(),
    "pages converted (silver_pages)": spark.table(pages_tbl).count(),
    "complete documents (silver_documents)": spark.table(docs_view).count(),
}
for name, n in counts.items():
    print(f"{n:>8}  {name}")
display(spark.table(failures_view).groupBy("stage", "retryable").count())
display(spark.table(failures_view).orderBy("stage", "path").limit(50))

## Notes

- **Rerun:** safe at any point. The stream skips recorded PDFs, the retry only takes open retryable failures, and the conversion only takes missing pages. Nothing already stored is paid for again.
- **A PDF that stays open:** look at `silver_failures`. A parse failure that is not retryable (over 500 pages, failed conversion) needs a different fix, for example splitting the PDF.
- **Redo the Markdown with a new prompt:** `DELETE FROM silver_pages`, then run. The parses are kept.
- **Full rebuild:** drop the three tables and the two views, delete the checkpoint folder `/Volumes/<catalog>/<schema>/docs/_checkpoints/silver_parse`, and run. Deleting only the tables leaves the checkpoint, so nothing would be reparsed.
- **Cost:** `ai_parse_document` is billed per page and `ai_query` per token. Every stage has a budget widget. Test with small budgets first.